# SAMap supplementary figure: panels a-g assembled

Places the seven panel files on the figure's 12-column grid (180 mm wide) and adds the panel letters, in capitals as
in the paper. Each panel is drawn by its own notebook in this directory, at its exact grid size:

| panel | notebook | shows |
|---|---|---|
| a | `sfig_samap_a_umap_mouse_pseudotime.ipynb` | mouse pseudotime on the joint SAMap UMAP |
| b | `sfig_samap_b_umap_transferred_pseudotime.ipynb` | pseudotime transferred onto human nuclei |
| c | `sfig_samap_c_umap_human_pseudotime.ipynb` | human pseudotime |
| d | `sfig_samap_d_edge_share.ipynb` | cross-species edge weight by target compartment |
| e | `sfig_samap_e_transferred_vs_human.ipynb` | transferred against human pseudotime, by compartment |
| f | `sfig_samap_f_sections_shuffled.ipynb` | transferred pseudotime in the human sections, observed and shuffled |
| g | `sfig_samap_g_permutation_statistics.ipynb` | observed statistics against 1,000 shuffles of each design |

**Inputs**, in `PANELS`: `sfig_samap_panel_<letter>.png`, `_600dpi.png`, `.svg` and `.provenance.json` of every
panel. **Outputs**, in `OUT`: `sfig_samap.png` (300 dpi), `sfig_samap_600dpi.png`, `sfig_samap.svg` (each panel's
SVG nested at its grid position, text kept as text) and `sfig_samap.provenance.json`, which merges the panels'
records. The notebook stops if the panels were drawn with different fonts.

**Running it.** Run all cells with this directory as the working directory: the shared modules next to the notebook
are imported from there. The cells above *Run* only define constants and functions. Paths are relative to the
repository root. Existing outputs are not overwritten unless `FORCE = True`.

The pipeline runs it with papermill (`workflow/rules/10c_samap_figures.smk`). By hand, from the repository root:

```bash
papermill notebooks/samap/sfig_samap_compose.ipynb \
    results/notebooks/samap/sfig_samap_compose.executed.ipynb \
    --cwd notebooks/samap -p FORCE True
```

Commit this notebook without outputs: the provenance record notes whether the notebook matches its commit.

In [ ]:
# Parameters. papermill -p NAME VALUE overrides them in a cell below this one.
# Paths are relative to the repository root; REPO = None finds the root with git from the working directory.
REPO = None
PANELS = "results/figures/samap_supp/panels"
OUT = "results/figures/samap_supp"
FORCE = False  # overwrite existing outputs
NOTEBOOK = "notebooks/samap/sfig_samap_compose.ipynb"  # this notebook, for the provenance record

In [ ]:
import json
import logging
import os
import re
import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.font_manager as fm
from IPython.display import Image as ShowImage, display
from PIL import Image, ImageDraw, ImageFont

import sfig_samap_style as style

# This notebook's messages in the cell output; other libraries stay at WARNING.
logging.basicConfig(format="%(message)s", stream=sys.stdout, force=True)
LOG = logging.getLogger("sfig_samap_compose")
LOG.setLevel(logging.INFO)

## Constants

In [ ]:
# Panel files carry lowercase letters; the figure labels panels with capitals, as the paper does (fig. S30A).
LETTERS = [panel["letter"] for panel in style.LAYOUT["panels"]]
LETTER_PT = 11

# Letters 1.5 mm in from the panel's left edge and 1 mm down, where every panel title leaves room for them.
LETTER_INSET_MM = (1.5, 1.0)

# SVG letters name the font; the PNG letters are drawn with the bold face of the font the panels used.
SVG_LETTER_FONTS = "Helvetica Neue, Helvetica, Arial, sans-serif"

## Assembly

In [ ]:
def bold_face(family: str):
    """(path, face index) of the bold face of `family`, for the PIL-drawn letters; DejaVu Sans Bold if none."""
    candidates = [
        entry for entry in fm.fontManager.ttflist
        if entry.name == family and entry.weight in (700, "bold") and entry.style == "normal"
    ]
    for entry in candidates:
        if not entry.fname.lower().endswith(".ttc"):
            return entry.fname, 0
        # A .ttc collection holds several faces; take the first whose style name is Bold.
        for index in range(32):
            try:
                face = ImageFont.truetype(entry.fname, 20, index=index)
            except OSError:
                break
            if face.getname()[1] == "Bold":
                return entry.fname, index
    return os.path.join(mpl.get_data_path(), "fonts", "ttf", "DejaVuSans-Bold.ttf"), 0


def compose_png(panel_files: dict, out_path: Path, dpi: int, letter_font: tuple) -> tuple:
    """Paste each panel PNG at its grid position on a white page and draw its letter; returns the page size (px)."""
    grid = style.grid_geometry(dpi)
    page = Image.new("RGB", (grid["width"], grid["height"]), "white")
    draw = ImageDraw.Draw(page)
    font = ImageFont.truetype(letter_font[0], int(LETTER_PT / 72 * dpi), index=letter_font[1])
    for letter in LETTERS:
        width, height = style.panel_size_px(letter, dpi)
        x, y = style.panel_origin_px(letter, dpi)
        panel = Image.open(panel_files[letter]).convert("RGBA")

        # Grid sizes are whole pixels at each resolution, so a 600-dpi panel can be a pixel off twice its
        # 300-dpi size.
        if panel.size != (width, height):
            panel = panel.resize((width, height))
        page.paste(panel, (x, y), panel)
        inset_x, inset_y = (int(mm / 25.4 * dpi) for mm in LETTER_INSET_MM)
        draw.text((x + inset_x, y + inset_y), letter.upper(), fill="black", font=font)
    page.save(out_path)
    return grid["width"], grid["height"]


def compose_svg(panel_dir: Path, out_path: Path) -> None:
    """Nest each panel SVG at its grid position; ids get the panel letter as prefix so panels cannot collide."""
    pt = 72 / style.DPI  # the grid is in 300-dpi pixels
    grid = style.grid_geometry()
    page_w, page_h = grid["width"] * pt, grid["height"] * pt
    parts = [
        f'<?xml version="1.0" encoding="utf-8"?>\n<svg xmlns="http://www.w3.org/2000/svg" '
        f'xmlns:xlink="http://www.w3.org/1999/xlink" width="{page_w:.2f}pt" height="{page_h:.2f}pt" '
        f'viewBox="0 0 {page_w:.2f} {page_h:.2f}">\n<rect width="100%" height="100%" fill="white"/>'
    ]
    for letter in LETTERS:
        body = (panel_dir / f"sfig_samap_panel_{letter}.svg").read_text()
        body = re.sub(r"<\?xml[^>]*\?>|<!DOCTYPE[^>]*>", "", body).strip()
        body = re.sub(r'id="([^"]+)"', rf'id="{letter}_\1"', body)
        body = re.sub(r'(xlink:href|href)="#([^"]+)"', rf'\1="#{letter}_\2"', body)
        body = re.sub(r"url\(#([^)]+)\)", rf"url(#{letter}_\1)", body)
        width = float(re.search(r'<svg[^>]*\swidth="([\d.]+)pt"', body).group(1))
        height = float(re.search(r'<svg[^>]*\sheight="([\d.]+)pt"', body).group(1))
        x, y = (value * pt for value in style.panel_origin_px(letter))
        nested = (f'<svg x="{x:.3f}" y="{y:.3f}" width="{width:.3f}" height="{height:.3f}" '
                  f'viewBox="0 0 {width} {height}" xmlns="http://www.w3.org/2000/svg" '
                  f'xmlns:xlink="http://www.w3.org/1999/xlink">')
        body = re.sub(r"<svg\b[^>]*>", nested, body, count=1)
        parts.append(f'<g id="panel_{letter}">{body}</g>')

        # Same place as the PNG letters: 1.5 mm in, baseline one cap height below 1 mm.
        parts.append(f'<text x="{x + 4.25:.2f}" y="{y + 11.41:.2f}" font-family="{SVG_LETTER_FONTS}" '
                     f'font-weight="bold" font-size="{LETTER_PT}" fill="black">{letter.upper()}</text>')
    parts.append("</svg>\n")
    out_path.write_text("\n".join(parts))

## Run

In [ ]:
repo = style.find_repo(REPO)
notebook = repo / NOTEBOOK
if not notebook.is_file():
    raise FileNotFoundError(f"{notebook} not found; set NOTEBOOK to this notebook's path")

panel_dir = repo / PANELS
out = repo / OUT
figure_files = {
    "png": out / "sfig_samap.png",
    "png_600dpi": out / "sfig_samap_600dpi.png",
    "svg": out / "sfig_samap.svg",
}
sidecar = out / "sfig_samap.provenance.json"
style.refuse_overwrite([*figure_files.values(), sidecar], FORCE)
out.mkdir(parents=True, exist_ok=True)

panels = {letter: style.panel_paths(panel_dir, letter) for letter in LETTERS}
missing = [str(path) for files in panels.values() for path in files.values() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Panel files missing; run the panel notebooks first: {missing}")
records = {letter: json.loads(files["provenance"].read_text()) for letter, files in panels.items()}

# One font across the figure: a panel drawn elsewhere with another font would not match its neighbors.
fonts = {letter: record["font"] for letter, record in records.items()}
if len(set(fonts.values())) != 1:
    raise ValueError(f"Panels were drawn with different fonts: {fonts}")
font = fonts["a"]
letter_font = bold_face(font)
LOG.info("Panel font %s; letters drawn with %s", font, letter_font)

In [ ]:
width_px, height_px = compose_png({letter: files["png"] for letter, files in panels.items()},
                                  figure_files["png"], style.DPI, letter_font)
compose_png({letter: files["png_600dpi"] for letter, files in panels.items()},
            figure_files["png_600dpi"], 600, letter_font)
compose_svg(panel_dir, figure_files["svg"])
LOG.info("Wrote %s; %d x %d px = %.0f x %.0f mm", [str(path) for path in figure_files.values()],
         width_px, height_px, width_px / style.PX_PER_MM, height_px / style.PX_PER_MM)

display(ShowImage(filename=str(figure_files["png"]), width=900))

In [ ]:
# The panels' facts under the keys of the figure's earlier provenance files.
sources = {}
for record in records.values():
    for entry in record["inputs"]:
        sources[entry["path"]] = entry

record = style.provenance(
    notebook,
    parameters={"REPO": str(repo), "PANELS": PANELS, "OUT": OUT, "FORCE": FORCE},
    inputs={f"panel_{letter}": files["provenance"] for letter, files in panels.items()},
    outputs=list(figure_files.values()),
    font=font,
)
record |= {
    "figure": str(figure_files["png"].resolve()),
    "size_px": [width_px, height_px],
    "size_mm": [round(width_px / style.PX_PER_MM, 1), round(height_px / style.PX_PER_MM, 1)],
    "letter_font": list(letter_font),
    "layout": style.LAYOUT,
    "panels": {
        letter: {key: panel_record[key] for key in ("notebook", "notebook_md5", "git", "written")}
        for letter, panel_record in records.items()
    },
    "sources": sorted(sources.values(), key=lambda entry: entry["path"]),
    "spearman_rho": records["e"]["spearman_rho"],
    "spatial_nuclei_drawn": records["f"]["spatial_nuclei_drawn"],
    "permutation_observed": records["g"]["permutation_observed"],
    "permutation_replicates": records["g"]["permutation_replicates"],
    "note": ("Mouse pseudotime: pt_harmonized_rank (2026_10_22_flagged_harmonized release) and its SAMap transfer of "
             "record. Human pseudotime: human_cells.csv pt_human (config pseudotime_of_record.human). Human sections "
             "mirrored left-right about their center. Every nucleus is drawn."),
}
style.write_json(sidecar, record)
LOG.info("Wrote %s", sidecar)